# Notebook: BCI_24_CarDet_Crit_Entrada_Pri
*********************************************************************************

## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_24_CarDet_Crit_Entrada_Pri.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011898507
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 12/08/2022
* Descripcion: selecciona el criterio de entrada principal segun jerarquia
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 25/04/2025 
* Descripción: Se modifica condicion para dejar la minima fecha de entrada de la operacion a cartera deteriorada.     
***************************************************************************

**************************************************************************
#### Mantención Nro: 2
* Autor: Gonzalo Arias (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 30/06/2025 
* Descripción: Se modifica flujo para asignar deterioro principal. Se utiliza
*              logica onpremise. Primero se deteriora considerando solo criterios Bci
*              segundo, se deteriora con criterios de filiales.
***************************************************************************


**************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 22/07/2025 
* Descripción: Se modifica condicion para irradiar las operaciones Hipotecarias y Cae entre si.
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_cartdet_crit_ent_crit
* {base_silver_x}.tbl_cd_d00_segmentado 
* {base_silver_x}.tbl_cd_d00_segmentado_pant
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_cartdet_crit_ent_prin
***************************************************************************


## Carga Dependencias

### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

## Parametría

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","03-Nombre BD Silver:")


fecha_x = dbutils.widgets.get("fecha_w")
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")


### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

In [0]:
# Calcula periodo en base a la fecha
periodo_x=fecha_x[:6]

print(f"Periodo: [periodo_x] {periodo_x}")

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
#Parametria interna notebook
p_crit_det = 1,7,8,9,10,11,12,13
p_periodo_evaluacion='p_anterior'
p_tio_esp='HIP','CAE'
p_ind_cartdet='D'

print(f"p_crit_det: {p_crit_det}")
print(f"p_periodo_evaluacion: {p_periodo_evaluacion}")
print(f"p_tio_esp: {p_tio_esp}")
print(f"p_ind_cartdet: {p_ind_cartdet}")


### [BCI] CALCULO DETERIORO BCI
--------------------------------------

#### Extrae criterios de deterioro de las operaciones BCI
--------------------------------------
- obtiene todos los deterioros calculados en proceso previos
- genera jerarquia (esta jerarquia debe estar en la tabla de parametros)
- son 3 jerarquias. criterio deterioro, grupo, y periodo evaluacion
- Esta jerarquia es para determinar el calculo del deterioro (no es lo mismo que el criterio que se informa x cliente)

In [0]:
paso_query20 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cartdet_crit_ent_crit_BCI AS
SELECT
  A.periodo_cierre       AS periodo_cierre,
  A.fecha_cierre         AS fecha_cierre,
  A.tipo_proceso         AS tipo_proceso,
  A.rut_cliente          AS rut_cliente,
  A.dv_rut_cliente       AS dv_rut_cliente,
  A.tipo_operacion       AS tipo_operacion,
  A.operacion            AS operacion,
  A.sistema              AS sistema,
  A.segmento             AS segmento,
  A.criterio_entrada     AS criterio_entrada,
  A.origen_deterioro     AS origen_deterioro,
  MIN(A.fecha_entrada) OVER(PARTITION BY A.operacion, A.sistema ORDER BY A.fecha_cierre ASC) AS fecha_entrada,
  A.grupo                AS grupo,
  A.periodo_evaluacion   AS periodo_evaluacion,
  CASE 
      WHEN A.criterio_entrada = 1 THEN 1
      WHEN A.criterio_entrada = 11 THEN 2
      WHEN A.criterio_entrada = 8 THEN 3
      WHEN A.criterio_entrada = 7 THEN 4
      WHEN A.criterio_entrada = 9 THEN 5
      WHEN A.criterio_entrada = 10 THEN 6
      WHEN A.criterio_entrada = 12 THEN 7
      WHEN A.criterio_entrada = 13 THEN 8      
      ELSE 99
  END                    AS jer_criterio_entrada,  
  CASE 
      WHEN trim(A.grupo) = 'BCI_Individual' THEN 1
      WHEN trim(A.grupo) = 'BCI_Grupal' THEN 2
      WHEN trim(A.grupo) = 'Factoring' THEN 3
      WHEN trim(A.grupo) = 'SSFF' THEN 4
      ELSE 99
  END                    AS jer_grupo,
  CASE 
      WHEN trim(A.periodo_evaluacion) = 'p_actual' THEN 1
      WHEN trim(A.periodo_evaluacion) = 'p_anterior' THEN 2
      ELSE 99
  END                    AS jer_periodo_evaluacion  
FROM
  {base_silver_x}.tbl_cd_cartdet_crit_ent_crit A
WHERE
    fecha_cierre = {fecha_x}  
AND periodo_cierre = {periodo_x}  
AND trim(A.grupo) IN ('BCI_Individual','BCI_Grupal') /*SOLO CRITERIOS BCI*/ 
""" 


In [0]:
sql_safe(paso_query20)

#### Obtiene criterio principal por operacion BCI
--------------------------------------
- selecciona el criterio principal segun jerarquia de los 3 campos


In [0]:
paso_query30 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cartdet_crit_ent_ope_prin_BCI AS
SELECT
  A.periodo_cierre           AS periodo_cierre,
  A.fecha_cierre             AS fecha_cierre,
  A.tipo_proceso             AS tipo_proceso,
  A.rut_cliente              AS rut_cliente,
  A.dv_rut_cliente           AS dv_rut_cliente,
  A.tipo_operacion           AS tipo_operacion,
  A.operacion                AS operacion,
  A.sistema                  AS sistema,
  A.segmento                 AS segmento,
  A.criterio_entrada         AS criterio_entrada,
  A.origen_deterioro         AS origen_deterioro,
  A.fecha_entrada            AS fecha_entrada,
  A.grupo                    AS grupo,
  A.periodo_evaluacion       AS periodo_evaluacion,
  A.jer_periodo_evaluacion   AS jer_periodo_evaluacion,
  A.jer_grupo                AS jer_grupo,
  A.jer_criterio_entrada     AS jer_criterio_entrada
  
FROM
  tmp_EXT_tbl_cartdet_crit_ent_crit_BCI A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.operacion, A.sistema ORDER BY  A.jer_criterio_entrada ASC, A.jer_periodo_evaluacion ASC, A.jer_grupo ASC) =1  
""" 


In [0]:
sql_safe(paso_query30)

#### Obtiene criterio principal del cliente BCI
--------------------------------------
- Selecciona el criterio principal segun jerarquia, para el cliente 
- Este criterio principal se usa para la irradiacion de deterioro. 
- Este criterio principal del cliente no es lo mismo que el criterio de deterioro del cliente que se informara en los archivos finales

In [0]:
paso_query40 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cartdet_crit_ent_cli_prin_BCI AS
SELECT
  A.periodo_cierre       AS periodo_cierre,
  A.fecha_cierre         AS fecha_cierre,
  A.tipo_proceso         AS tipo_proceso,
  A.rut_cliente          AS rut_cliente,
  A.dv_rut_cliente       AS dv_rut_cliente,
  A.criterio_entrada     AS criterio_entrada

FROM
  tmp_RES_tbl_cartdet_crit_ent_ope_prin_BCI A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.rut_cliente ORDER BY A.jer_periodo_evaluacion ASC, A.jer_grupo ASC, A.jer_criterio_entrada ASC) =1  
""" 


In [0]:
sql_safe(paso_query40)

#### Obtiene operaciones deterioradas con criterio propio BCI
--------------------------------------
- genera salida de operaciones deterioradas con criterio propio o heredado del mes anterior


In [0]:
paso_query50 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_det_prin_BCI AS
SELECT 
    U.periodo_cierre	                      AS periodo_cierre	                  
   ,U.fecha_cierre	                      AS fecha_cierre	                  
   ,U.tipo_proceso	                      AS tipo_proceso	                  
   ,U.rut_cliente	                         AS rut_cliente	                  
   ,U.dv_rut_cliente	                      AS dv_rut_cliente	                  
   ,U.tipo_operacion	                      AS tipo_operacion	                  
   ,U.operacion	                         AS operacion	                      
   ,U.sistema	                            AS sistema	                      
   ,U.segmento	                            AS segmento	                      
   ,U.criterio_entrada	                   AS criterio_entrada	              
   ,U.origen_deterioro	                   AS origen_deterioro	              
   ,U.fecha_entrada	                      AS fecha_entrada	                  
   ,U.grupo	                               AS grupo	                          
   ,U.periodo_evaluacion                   AS periodo_evaluacion               
   ,A.criterio_entrada                     AS criterio_entrada_cliente
FROM   
   tmp_RES_tbl_cartdet_crit_ent_ope_prin_BCI U,
   tmp_RES_tbl_cartdet_crit_ent_cli_prin_BCI A
WHERE
   U.rut_cliente = A.rut_cliente
"""  

In [0]:
sql_safe(paso_query50)

#### Obtiene operaciones de clientes deteriorados BCI
--------------------------------------
- Obtiene todas las operaciones de los clientes con deterioro propio calculados en pasos previos



In [0]:
paso_query60 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_cli_BCI AS
SELECT
    U.periodo_cierre	                    AS periodo_cierre	                  
   ,U.fecha_cierre	                         AS fecha_cierre	                  
   ,U.tipo_proceso	                         AS tipo_proceso	                  
   ,U.rut_cliente	                         AS rut_cliente	                  
   ,U.dv_rut_cliente	                    AS dv_rut_cliente	                  
   ,U.tipo_operacion	                    AS tipo_operacion	                  
   ,U.operacion	                         AS operacion	                      
   ,U.sistema	                              AS sistema	                      
   ,U.segmento	                              AS segmento	                      
FROM
     {base_silver_x}.tbl_cd_d00_segmentado U,
     tmp_RES_tbl_cartdet_crit_ent_cli_prin_BCI  A
WHERE
     U.rut_cliente = A.rut_cliente
"""  

In [0]:
sql_safe(paso_query60)

#### Operaciones irradiadas 1: operaciones que no son HIP ni CAE
--------------------------------------
- si cliente tiene deterioro, las operaciones sin deterioro propio son irradiadas segun el criterio del cliente
- excepciones: operaciones hipotecarias vivienda (HIP) y creditos con aval del estado (CAE)


In [0]:
paso_query70 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_det_prin_irra_1_BCI AS
SELECT
    U.periodo_cierre	                    AS periodo_cierre	                  
   ,U.fecha_cierre	                         AS fecha_cierre	                  
   ,U.tipo_proceso	                         AS tipo_proceso	                  
   ,U.rut_cliente	                         AS rut_cliente	                  
   ,U.dv_rut_cliente	                    AS dv_rut_cliente	                  
   ,U.tipo_operacion	                    AS tipo_operacion	                  
   ,U.operacion	                         AS operacion	                      
   ,U.sistema	                              AS sistema	                      
   ,U.segmento	                              AS segmento	                      
   ,A.criterio_entrada	                    AS criterio_entrada	              
   ,5               	                    AS origen_deterioro	              
   ,U.fecha_cierre	                         AS fecha_entrada	                  
   ,'Bci_Irradiacion'	                    AS grupo	                          
   ,'p_actual'                               AS periodo_evaluacion               
   ,A.criterio_entrada                       AS criterio_entrada_cliente         
FROM
     tmp_RES_cd_cartdet_ope_cli_BCI U
INNER JOIN
     tmp_RES_tbl_cartdet_crit_ent_cli_prin_BCI  A
ON U.rut_cliente = A.rut_cliente     
LEFT JOIN 
      tmp_RES_cd_cartdet_ope_det_prin_BCI B
ON U.operacion = B.operacion AND U.sistema = B.sistema
WHERE
     B.operacion IS NULL /*que no tenga deterioro propio*/
AND  substring(U.tipo_operacion,1,3) NOT IN {p_tio_esp}  /*no irradie hipotecario vivienda ni cae*/
"""

In [0]:
sql_safe(paso_query70)

#### Operaciones irradiadas 2: Para operaciones CAE
--------------------------------------
- Los cae deteriorados, NO irradian hipotecarios
- Obtiene todos los clientes con criterio deterioro propio de mora paras las operaciones CAE

In [0]:
paso_query75 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_cli_cae_det_prp_BCI AS
select 
A.*
from tmp_RES_cd_cartdet_ope_det_prin_BCI A
where 
substring(trim(A.tipo_operacion),1,3)='CAE' and
A.origen_deterioro=1 and
A.criterio_entrada_cliente=8 
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.rut_cliente ORDER BY A.periodo_cierre asc) =1  
"""


In [0]:
sql_safe(paso_query75) 

In [0]:
paso_query80 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_det_prin_irra_2_BCI AS
SELECT
    U.periodo_cierre	                 AS periodo_cierre	                  
   ,U.fecha_cierre	                      AS fecha_cierre	                  
   ,U.tipo_proceso	                      AS tipo_proceso	                  
   ,U.rut_cliente	                      AS rut_cliente	                  
   ,U.dv_rut_cliente	                 AS dv_rut_cliente	                  
   ,U.tipo_operacion	                 AS tipo_operacion	                  
   ,U.operacion	                      AS operacion	                      
   ,U.sistema	                           AS sistema	                      
   ,U.segmento	                           AS segmento	                      
   ,A.criterio_entrada	                 AS criterio_entrada	              
   ,5               	                 AS origen_deterioro	              
   ,U.fecha_cierre	                      AS fecha_entrada	                  
   ,'Bci_Irradiacion'	                 AS grupo	                          
   ,'p_actual'                            AS periodo_evaluacion               
   ,A.criterio_entrada                    AS criterio_entrada_cliente         
FROM
     tmp_RES_cd_cartdet_ope_cli_BCI U
INNER JOIN
     tmp_RES_tbl_cartdet_crit_ent_cli_prin_BCI  A
ON U.rut_cliente = A.rut_cliente       
INNER JOIN
     tmp_RES_cd_cartdet_cli_cae_det_prp_BCI  B  /*SOLO CLIENTES DETERIORO PROPIO CAE*/
ON U.rut_cliente = B.rut_cliente     
LEFT JOIN
      tmp_RES_cd_cartdet_ope_det_prin_BCI C
ON U.operacion = C.operacion AND U.sistema = C.sistema
WHERE
    C.operacion IS NULL /*que no tenga deterioro propio*/
AND substring(trim(U.tipo_operacion),1,3) <> 'HIP'  /*NO IRRADIA A HIP VIV*/
AND coalesce(A.criterio_entrada,0) = 8 /*criterio deterioro 8 [hipotecario o cae]*/
"""

In [0]:
sql_safe(paso_query80) 

#### Operaciones irradiadas 3: Para operaciones  HIP  
--------------------------------------
- Los hipotecarios deteriorados, NO irradian a los cae
- Obtiene todos los clientes con criterio deterioro propio de mora paras las operaciones hipotecaria vivienda

In [0]:
paso_query81 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_cli_hip_det_prp_BCI AS
select 
A.*
from tmp_RES_cd_cartdet_ope_det_prin_BCI A
where 
substring(trim(A.tipo_operacion),1,3)='HIP' and
A.origen_deterioro=1 and
A.criterio_entrada_cliente=8  
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.rut_cliente ORDER BY A.periodo_cierre asc) =1  
"""

In [0]:
sql_safe(paso_query81)

In [0]:
paso_query85 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_det_prin_irra_3_BCI AS
SELECT
    U.periodo_cierre	                 AS periodo_cierre	                  
   ,U.fecha_cierre	                   AS fecha_cierre	                  
   ,U.tipo_proceso	                   AS tipo_proceso	                  
   ,U.rut_cliente	                     AS rut_cliente	                  
   ,U.dv_rut_cliente	                 AS dv_rut_cliente	                  
   ,U.tipo_operacion	                 AS tipo_operacion	                  
   ,U.operacion	                       AS operacion	                      
   ,U.sistema	                         AS sistema	                      
   ,U.segmento	                       AS segmento	                      
   ,A.criterio_entrada	               AS criterio_entrada	              
   ,5               	                 AS origen_deterioro	              
   ,U.fecha_cierre	                   AS fecha_entrada	                  
   ,'Bci_Irradiacion'	                 AS grupo	                          
   ,'p_actual'                         AS periodo_evaluacion               
   ,A.criterio_entrada                 AS criterio_entrada_cliente         
FROM
     tmp_RES_cd_cartdet_ope_cli_BCI U
INNER JOIN
     tmp_RES_tbl_cartdet_crit_ent_cli_prin_BCI  A
ON U.rut_cliente = A.rut_cliente            
INNER JOIN
     tmp_RES_cd_cartdet_cli_hip_det_prp_BCI  B  /*SOLO CLIENTES DETERIORO PROPIO HIP VIV*/
ON U.rut_cliente = B.rut_cliente     
LEFT JOIN
      tmp_RES_cd_cartdet_ope_det_prin_BCI C
ON U.operacion = C.operacion AND U.sistema = C.sistema
WHERE
    C.operacion IS NULL /*que no tenga deterioro propio*/
AND substring(trim(U.tipo_operacion),1,3) <> 'CAE'  /*NO IRRADIA CAE*/
AND coalesce(A.criterio_entrada,0) = 8 /*criterio deterioro 8 [hipotecario o cae]*/
"""

In [0]:
sql_safe(paso_query85) 

#### Operaciones irradiadas 4: Para clientes LIR
--------------------------------------
- Los lir deteriorados, irradian a todos

In [0]:
paso_query87 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_det_prin_irra_4_BCI AS
SELECT
    U.periodo_cierre	                    AS periodo_cierre	                  
   ,U.fecha_cierre	                      AS fecha_cierre	                  
   ,U.tipo_proceso	                      AS tipo_proceso	                  
   ,U.rut_cliente	                        AS rut_cliente	                  
   ,U.dv_rut_cliente	                    AS dv_rut_cliente	                  
   ,U.tipo_operacion	                    AS tipo_operacion	                  
   ,U.operacion	                          AS operacion	                      
   ,U.sistema	                            AS sistema	                      
   ,U.segmento	                          AS segmento	                      
   ,A.criterio_entrada	                  AS criterio_entrada	              
   ,5               	                    AS origen_deterioro	              
   ,U.fecha_cierre	                      AS fecha_entrada	                  
   ,'Bci_Irradiacion'	                    AS grupo	                          
   ,'p_actual'                            AS periodo_evaluacion               
   ,A.criterio_entrada                    AS criterio_entrada_cliente         
FROM
     tmp_RES_cd_cartdet_ope_cli_BCI U
INNER JOIN
     tmp_RES_tbl_cartdet_crit_ent_cli_prin_BCI  A  /*CLIENTES CON CRITERIO PROPIO*/
ON U.rut_cliente = A.rut_cliente     
LEFT JOIN
      tmp_RES_cd_cartdet_ope_det_prin_BCI B
ON U.operacion = B.operacion AND U.sistema = B.sistema
WHERE
    B.operacion IS NULL /*que no tenga deterioro propio*/
AND coalesce(A.criterio_entrada,0) = 11 /*criterio deterioro 11 [cliente Lir]*/
"""

In [0]:
sql_safe(paso_query87) 

#### Genera salida de deterioro propios e irradiados BCI
------------------
* Une las salida de deterioro propio + irradiados + irradiados hip, cae


In [0]:
paso_query90 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cd_cartdet_crit_ent_BCI AS
SELECT * FROM tmp_RES_cd_cartdet_ope_det_prin_BCI
UNION
SELECT * FROM tmp_RES_cd_cartdet_ope_det_prin_irra_1_BCI
UNION
SELECT * FROM tmp_RES_cd_cartdet_ope_det_prin_irra_2_BCI
UNION
SELECT * FROM tmp_RES_cd_cartdet_ope_det_prin_irra_3_BCI
UNION
SELECT * FROM tmp_RES_cd_cartdet_ope_det_prin_irra_4_BCI
"""  

In [0]:
sql_safe(paso_query90)

### [FILIALES] CALCULO DETERIORO FILIALES
--------------------------------------

#### Extrae criterios de deterioro de las operaciones FILIALES
--------------------------------------
- obtiene todos los deterioros calculados en proceso previos
- genera jerarquia (esta jerarquia debe estar en la tabla de parametros)
- son 3 jerarquias. criterio deterioro, grupo, y periodo evaluacion
- Esta jerarquia es para determinar el calculo del deterioro (no es lo mismo que el criterio que se informa x cliente)

In [0]:
paso_query110 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cartdet_crit_ent_crit_FIL AS
SELECT
  A.periodo_cierre       AS periodo_cierre,
  A.fecha_cierre         AS fecha_cierre,
  A.tipo_proceso         AS tipo_proceso,
  A.rut_cliente          AS rut_cliente,
  A.dv_rut_cliente       AS dv_rut_cliente,
  A.tipo_operacion       AS tipo_operacion,
  A.operacion            AS operacion,
  A.sistema              AS sistema,
  A.segmento             AS segmento,
  A.criterio_entrada     AS criterio_entrada,
  A.origen_deterioro     AS origen_deterioro,
  MIN(A.fecha_entrada) OVER(PARTITION BY A.operacion, A.sistema ORDER BY A.fecha_cierre ASC) AS fecha_entrada,
  A.grupo                AS grupo,
  A.periodo_evaluacion   AS periodo_evaluacion,
  CASE 
      WHEN A.criterio_entrada = 1 THEN 1
      WHEN A.criterio_entrada = 11 THEN 2
      WHEN A.criterio_entrada = 8 THEN 3
      WHEN A.criterio_entrada = 7 THEN 4
      WHEN A.criterio_entrada = 9 THEN 5
      WHEN A.criterio_entrada = 10 THEN 6
      WHEN A.criterio_entrada = 12 THEN 7
      WHEN A.criterio_entrada = 13 THEN 8      
      ELSE 99
  END                    AS jer_criterio_entrada,  
  CASE 
      WHEN trim(A.grupo) = 'BCI_Individual' THEN 1
      WHEN trim(A.grupo) = 'BCI_Grupal' THEN 2
      WHEN trim(A.grupo) = 'Factoring' THEN 3
      WHEN trim(A.grupo) = 'SSFF' THEN 4
      ELSE 99
  END                    AS jer_grupo,
  CASE 
      WHEN trim(A.periodo_evaluacion) = 'p_actual' THEN 1
      WHEN trim(A.periodo_evaluacion) = 'p_anterior' THEN 2
      ELSE 99
  END                    AS jer_periodo_evaluacion  
FROM
  {base_silver_x}.tbl_cd_cartdet_crit_ent_crit A
WHERE
    fecha_cierre = {fecha_x}  
AND periodo_cierre = {periodo_x}  
AND trim(A.grupo) IN ('Factoring','SSFF') /*SOLO CRITERIOS FILIALES*/ 
""" 


In [0]:
sql_safe(paso_query110) 

#### Obtiene criterio principal por operacion FILIALES
--------------------------------------
- selecciona el criterio principal segun jerarquia de los 3 campos


In [0]:
paso_query115 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cartdet_crit_ent_ope_prin_FIL AS
SELECT
  A.periodo_cierre           AS periodo_cierre,
  A.fecha_cierre             AS fecha_cierre,
  A.tipo_proceso             AS tipo_proceso,
  A.rut_cliente              AS rut_cliente,
  A.dv_rut_cliente           AS dv_rut_cliente,
  A.tipo_operacion           AS tipo_operacion,
  A.operacion                AS operacion,
  A.sistema                  AS sistema,
  A.segmento                 AS segmento,
  A.criterio_entrada         AS criterio_entrada,
  A.origen_deterioro         AS origen_deterioro,
  A.fecha_entrada            AS fecha_entrada,
  A.grupo                    AS grupo,
  A.periodo_evaluacion       AS periodo_evaluacion,
  A.jer_periodo_evaluacion   AS jer_periodo_evaluacion,
  A.jer_grupo                AS jer_grupo,
  A.jer_criterio_entrada     AS jer_criterio_entrada
  
FROM
  tmp_EXT_tbl_cartdet_crit_ent_crit_FIL A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.operacion, A.sistema ORDER BY A.jer_periodo_evaluacion ASC, A.jer_grupo ASC, A.jer_criterio_entrada ASC) =1  
""" 


In [0]:
sql_safe(paso_query115) 

#### Obtiene criterio principal del cliente FILIALES
--------------------------------------
- Selecciona el criterio principal segun jerarquia, para el cliente 
- Este criterio principal se usa para la irradiacion de deterioro. 
- Este criterio principal del cliente no es lo mismo que el criterio de deterioro del cliente que se informara en los archivos finales

In [0]:
paso_query120 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_tbl_cartdet_crit_ent_cli_prin_FIL AS
SELECT
  A.periodo_cierre       AS periodo_cierre,
  A.fecha_cierre         AS fecha_cierre,
  A.tipo_proceso         AS tipo_proceso,
  A.rut_cliente          AS rut_cliente,
  A.dv_rut_cliente       AS dv_rut_cliente,
  A.criterio_entrada     AS criterio_entrada

FROM
  tmp_RES_tbl_cartdet_crit_ent_ope_prin_FIL A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.rut_cliente ORDER BY A.jer_periodo_evaluacion ASC, A.jer_grupo ASC, A.jer_criterio_entrada ASC) =1  
""" 



In [0]:
sql_safe(paso_query120) 

#### Obtiene operaciones deterioradas con criterio propio FILIALES
--------------------------------------
- genera salida de operaciones deterioradas con criterio propio o heredado del mes anterior


In [0]:
paso_query125 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_det_prin_FIL AS
SELECT 
    U.periodo_cierre	                    AS periodo_cierre	                  
   ,U.fecha_cierre	                      AS fecha_cierre	                  
   ,U.tipo_proceso	                      AS tipo_proceso	                  
   ,U.rut_cliente	                        AS rut_cliente	                  
   ,U.dv_rut_cliente	                    AS dv_rut_cliente	                  
   ,U.tipo_operacion	                    AS tipo_operacion	                  
   ,U.operacion	                          AS operacion	                      
   ,U.sistema	                            AS sistema	                      
   ,U.segmento	                          AS segmento	                      
   ,U.criterio_entrada	                  AS criterio_entrada	              
   ,U.origen_deterioro	                  AS origen_deterioro	              
   ,U.fecha_entrada	                      AS fecha_entrada	                  
   ,U.grupo	                              AS grupo	                          
   ,U.periodo_evaluacion                  AS periodo_evaluacion               
   ,A.criterio_entrada                    AS criterio_entrada_cliente
FROM   
   tmp_RES_tbl_cartdet_crit_ent_ope_prin_FIL U,
   tmp_RES_tbl_cartdet_crit_ent_cli_prin_FIL A
WHERE
   U.rut_cliente = A.rut_cliente
"""  

In [0]:
sql_safe(paso_query125) 

#### Obtiene operaciones de clientes deteriorados FILIALES
--------------------------------------
- Obtiene todas las operaciones de los clientes con deterioro propio calculados en pasos previos



In [0]:
paso_query130 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_cli_FIL AS
SELECT
    U.periodo_cierre	                 AS periodo_cierre	                  
   ,U.fecha_cierre	                   AS fecha_cierre	                  
   ,U.tipo_proceso	                   AS tipo_proceso	                  
   ,U.rut_cliente	                     AS rut_cliente	                  
   ,U.dv_rut_cliente	                 AS dv_rut_cliente	                  
   ,U.tipo_operacion	                 AS tipo_operacion	                  
   ,U.operacion	                       AS operacion	                      
   ,U.sistema	                         AS sistema	                      
   ,U.segmento	                       AS segmento	                      
FROM
     {base_silver_x}.tbl_cd_d00_segmentado U,
     tmp_RES_tbl_cartdet_crit_ent_cli_prin_FIL  A
WHERE
     U.rut_cliente = A.rut_cliente
"""  

In [0]:
sql_safe(paso_query130) 

#### Operaciones irradiadas 1: operaciones que no son HIP ni CAE
--------------------------------------
- si cliente tiene deterioro, las operaciones sin deterioro propio son irradiadas segun el criterio del cliente
- excepciones: operaciones hipotecarias vivienda (HIP) y creditos con aval del estado (CAE)


In [0]:
paso_query135 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_cd_cartdet_ope_det_prin_irra_1_FIL AS
SELECT
    U.periodo_cierre	                 AS periodo_cierre	                  
   ,U.fecha_cierre	                      AS fecha_cierre	                  
   ,U.tipo_proceso	                      AS tipo_proceso	                  
   ,U.rut_cliente	                      AS rut_cliente	                  
   ,U.dv_rut_cliente	                 AS dv_rut_cliente	                  
   ,U.tipo_operacion	                 AS tipo_operacion	                  
   ,U.operacion	                      AS operacion	                      
   ,U.sistema	                           AS sistema	                      
   ,U.segmento	                           AS segmento	                      
   ,A.criterio_entrada	                 AS criterio_entrada	              
   ,6               	                 AS origen_deterioro	              
   ,U.fecha_cierre	                      AS fecha_entrada	                  
   ,'Fil_Irradiacion'	                 AS grupo	                          
   ,'p_actual'                            AS periodo_evaluacion               
   ,A.criterio_entrada                    AS criterio_entrada_cliente         
FROM
     tmp_RES_cd_cartdet_ope_cli_FIL U
LEFT JOIN
     tmp_RES_tbl_cartdet_crit_ent_cli_prin_FIL  A
ON U.rut_cliente = A.rut_cliente     
LEFT JOIN
      tmp_RES_cd_cartdet_ope_det_prin_FIL B
ON U.operacion = B.operacion AND U.sistema = B.sistema
WHERE
     B.operacion IS NULL /*que no tenga deterioro propio*/
AND  substring(U.tipo_operacion,1,3) NOT IN {p_tio_esp}  /*no irradie hipotecario vivienda ni cae*/
"""

In [0]:
sql_safe(paso_query135) 

#### Une registros de deterioro propios e irradiados FILIALES
------------------
* Une las salida de deterioro propio FILIALES + irradiados FILIALES


In [0]:
paso_query140 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cd_cartdet_crit_ent_FIL AS
SELECT * FROM tmp_RES_cd_cartdet_ope_det_prin_FIL
UNION
SELECT * FROM tmp_RES_cd_cartdet_ope_det_prin_irra_1_FIL
"""  

In [0]:
sql_safe(paso_query140) 

### [Salida Temporal]  Genera salida deterioro BCI y Filiales
------------------
* une informacion de deterioro de BCI y Filiales


#### Une informacion de deterioro de BCI y Filiales


In [0]:
paso_query250 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cd_cartdet_crit_ent_prin_1 AS
SELECT A.*, 1 AS IND FROM  tmp_tbl_cd_cartdet_crit_ent_BCI A
UNION  
SELECT B.*, 2 AS IND FROM  tmp_tbl_cd_cartdet_crit_ent_FIL B
"""  

In [0]:
sql_safe(paso_query250)

#### Asigna Jerarquia al Criterio de Deterioro
---
- considera todos los deterioros, bci y filiales, del mes actual y del mes anterior
- obtiene la minima fecha de entrada a deterioro
- es distinto al deterior por cliente que se usa para deteriorar


In [0]:
paso_query255= f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cd_cartdet_crit_ent_prin_2 AS
SELECT
  A.periodo_cierre,
  A.fecha_cierre,
  A.tipo_proceso,
  A.rut_cliente,
  A.dv_rut_cliente,
  A.tipo_operacion,
  A.operacion,
  A.sistema,
  A.segmento,
  A.criterio_entrada,
  A.origen_deterioro,
  MIN(A.fecha_entrada) OVER(PARTITION BY A.operacion, A.sistema ORDER BY A.fecha_cierre ASC) AS fecha_entrada,
  A.grupo,
  A.periodo_evaluacion,
  A.criterio_entrada_cliente,
  A.IND,
  CASE 
      WHEN A.criterio_entrada = 1 THEN 1
      WHEN A.criterio_entrada = 11 THEN 2
      WHEN A.criterio_entrada = 8 THEN 3
      WHEN A.criterio_entrada = 7 THEN 4
      WHEN A.criterio_entrada = 9 THEN 5
      WHEN A.criterio_entrada = 10 THEN 6
      WHEN A.criterio_entrada = 12 THEN 7
      WHEN A.criterio_entrada = 13 THEN 8      
      ELSE 99
  END                                         AS jer_criterio_entrada_show
FROM
  tmp_tbl_cd_cartdet_crit_ent_prin_1 A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.operacion, A.sistema ORDER BY A.IND ASC) =1    
""" 


In [0]:
sql_safe(paso_query255)

#### Calcula criterio deterioro por cliente
---
- considera todos los deterioros, bci y filiales, del mes actual y del mes anterior


In [0]:
paso_query257= f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cd_cartdet_crit_ent_cli_show AS
select 
 A.periodo_cierre
,A.fecha_cierre
,A.tipo_proceso
,A.rut_cliente
,A.criterio_entrada AS criterio_entrada_show
from 
  tmp_tbl_cd_cartdet_crit_ent_prin_2 A
QUALIFY  ROW_NUMBER() OVER(PARTITION BY A.rut_cliente ORDER BY A.jer_criterio_entrada_show ASC) =1      
""" 

In [0]:
sql_safe(paso_query257)

#### Genera tabla temporal final con datos de salida
---



In [0]:
paso_query260= f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cd_cartdet_crit_ent_prin AS
select 
 A.periodo_cierre
,A.fecha_cierre
,A.tipo_proceso
,A.rut_cliente
,A.dv_rut_cliente
,A.tipo_operacion
,A.operacion
,A.sistema
,A.segmento
,A.criterio_entrada
,A.origen_deterioro
,A.fecha_entrada
,A.grupo
,A.periodo_evaluacion
,A.criterio_entrada_cliente
,B.criterio_entrada_show
from 
  tmp_tbl_cd_cartdet_crit_ent_prin_2 A,
  tmp_tbl_cd_cartdet_crit_ent_cli_show B
where
	A.rut_cliente = B.rut_cliente
""" 

In [0]:
sql_safe(paso_query260)

## Carga Tablas de Salidas
--------------------------------------
* carga resultados a tablas de salidas del notebook

### Carga Tabla Evaluacion 


#### Reproceso (Elimina registros en caso de reprocesos). Tabla no es historica

In [0]:
paso_query300 = f""" TRUNCATE TABLE {base_silver_x}.tbl_cd_cartdet_crit_ent_prin  """

In [0]:
sql_safe(paso_query300)

#### Inserta Registros tabla salida

In [0]:
paso_query310 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_crit_ent_prin
SELECT 
    IFNULL(periodo_cierre,190001),
    IFNULL(fecha_cierre,19000101),
    IFNULL(tipo_proceso,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(segmento,' '),
    IFNULL(criterio_entrada,0),
    IFNULL(origen_deterioro,0),
    IFNULL(fecha_entrada,19000101),
    IFNULL(grupo,' '),
    IFNULL(periodo_evaluacion,' '),
    IFNULL(criterio_entrada_show,0)
FROM
    tmp_tbl_cd_cartdet_crit_ent_prin  
QUALIFY  ROW_NUMBER() OVER(PARTITION BY operacion, sistema ORDER BY  fecha_cierre ASC) =1 
"""  


In [0]:
sql_safe(paso_query310)

##Estadisticas tabla salida

In [0]:
%sql
SELECT
fecha_cierre,
criterio_entrada,
CASE 
  WHEN criterio_entrada=1 THEN 'CLASIFICACION_DETERIORO'
  WHEN criterio_entrada=7 THEN 'MOROSIDAD_NO_HIPCAE'
  WHEN criterio_entrada=8 THEN 'MOROSIDAD_HIPCAE'
  WHEN criterio_entrada=9 THEN 'RENEGOCIADO'
  WHEN criterio_entrada=10 THEN 'REESTRUCTURACION_FORZOSA'
  WHEN criterio_entrada=11 THEN 'LIR'
  WHEN criterio_entrada=12 THEN 'SSFF'
  WHEN criterio_entrada=13 THEN 'FACTORING'
  ELSE 'NO_IDENTIFICADO'    
END                    AS des_criterio_entrada,
COUNT(1) AS CANT_REG
FROM ${bci.dbnamesilver}.tbl_cd_cartdet_crit_ent_prin
GROUP BY 1,2,3
ORDER BY 1,2,3


## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")